# Hyperparameter tuning

In [0]:

from sklearn.datasets import load_iris
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load data
iris = load_iris()
X, y = iris.data, iris.target

# Reserve an untouched test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Define hyperparameter combinations
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [2, 3, 5, None],
    "min_samples_split": [2, 5, 10]
}

model = RandomForestClassifier(random_state=42)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Search for best hyperparameters
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

# Train and tune on training data only
grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation accuracy:")
print(f"{grid_search.best_score_:.4f}")

# Evaluate the selected model on unseen test data
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

print("\nTest accuracy:")
print(f"{accuracy_score(y_test, predictions):.4f}")


### ML workflow workflows 

In [0]:

import mlflow

with mlflow.start_run(run_name="Tuned Random Forest"):

    # Record best hyperparameters
    mlflow.log_params(grid_search.best_params_)

    # Record evaluation metrics
    mlflow.log_metric(
        "cv_accuracy",
        grid_search.best_score_
    )

    mlflow.log_metric(
        "test_accuracy",
        accuracy_score(y_test, predictions)
    )

    print("Tuned Random Forest logged to MLflow")


## Save Trained Model to MLFlow

In [0]:

import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score
from mlflow.models import infer_signature

# Retrieve the tuned model
best_model = grid_search.best_estimator_

# Predictions
predictions = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, predictions)

# Describe the model's input/output format
signature = infer_signature(
    X_test,
    predictions
)

# Start a new MLflow run
with mlflow.start_run(
    run_name="Iris Random Forest - Saved Model"
) as run:

    # Log hyperparameters
    mlflow.log_params(grid_search.best_params_)

    # Log metrics
    mlflow.log_metric(
        "cv_accuracy",
        grid_search.best_score_
    )
    mlflow.log_metric(
        "test_accuracy",
        test_accuracy
    )

    # Save the trained model as an MLflow artifact
    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="iris_random_forest",
        signature=signature
    )

    run_id = run.info.run_id

print("Model saved successfully!")
print("Run ID:", run_id)
print("Test accuracy:", test_accuracy)


## Reload from MLFlow  

In [0]:

import mlflow.sklearn

# Load the model from the run
model_uri = f"runs:/{run_id}/iris_random_forest"

loaded_model = mlflow.sklearn.load_model(model_uri)

print("Model loaded successfully!")
print("Model type:", type(loaded_model))


## Make predictions with reloaded Model


In [0]:

import numpy as np
from sklearn.datasets import load_iris

iris = load_iris()

# New flower measurements
# sepal length, sepal width, petal length, petal width
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])

prediction = loaded_model.predict(new_flower)

print("Predicted class:", prediction[0])
print("Predicted flower:", iris.target_names[prediction[0]])


## Verify the loaded model produces identical predictions


In [0]:

import numpy as np

original_predictions = best_model.predict(X_test)
loaded_predictions = loaded_model.predict(X_test)

assert np.array_equal(
    original_predictions,
    loaded_predictions
), "Model predictions do not match!"

print("SUCCESS: All predictions match!")
